### Trajectory data of a small molecular system: **alanine dipeptide**

Note: You can easily find pictures of the system on the internet. 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

## Load the data

**System**: The molecular system is alanine dipeptide. It contains 22 atoms in total. 

**Data**:

* The trajectory was simulated using molecular dynamics simulation.

* For simplicity, the data file contains the trajectory of 5 selected atoms of the system. 

* There are 150000 states in the trajectory dataset. Each state consists of the x,y,z coordinates of 5 atoms (dimension: $5\times 3$). 

In [ ]:
traj = np.load('dipeptide.npy')
print (traj.shape)

In [ ]:
print (traj[0])

### dihedral angles

The configuration of the system is determined by two dihedral angles $\phi$ and $\psi$ of the system. 

* The dihedral angle $\phi$ is a function of the first 4 atoms (i.e. atoms with indices 0,1,2,3).

* The dihedral angle $\psi$ is a function of the last 4 atoms (i.e. atoms with indices 1,2,3,4).


In [ ]:
# dihedral angle is a function of 4 atoms
def dihedral(x):
    r12 = x[:, 1, :] - x[:, 0, :]
    r23 = x[:, 2, :] - x[:, 1, :]
    r34 = x[:, 3, :] - x[:, 2, :]
    n1 = np.cross(r12, r23, axis=-1)
    n2 = np.cross(r23, r34, axis=-1)
    cos_phi = np.sum(n1 * n2, axis=1)
    sin_phi = np.sum(n1 * r34, axis=1) * np.linalg.norm(r23, axis=1)
    return np.arctan2(sin_phi, cos_phi) * (180 / np.pi)

#computes the phi angle
def angle_phi(x, atom_indices=[0,1,2,3]):
    return dihedral(x[:, atom_indices, :])

#computes the psi angle
def angle_psi(x, atom_indices=[1, 2, 3, 4]):
    return dihedral(x[:, atom_indices, :])

phi = angle_phi(traj)
psi = angle_psi(traj)

print (phi.shape, psi.shape)

### projection onto dihedral angle space

The system favors certain configurations, as can be seen from the histogram of dihedral angles.  

In [ ]:
fig = plt.figure(figsize=(12,5))

# scatter plot

ax = fig.add_subplot(1, 2, 1)
ax.scatter(phi, psi)
ax.set_aspect('equal')
ax.set_xlim([-180, 180])
ax.set_ylim([-180, 180])
ax.set_xlabel(r'$\phi$',fontsize=15)
ax.set_ylabel(r'$\psi$',fontsize=15, rotation=0)
ax.tick_params(axis='both', labelsize=12)
ax.set_xticks([-150, -100, -50, 0, 50, 100, 150])
ax.set_yticks([-150, -100, -50, 0, 50, 100, 150])
ax.set_title('scatter plot')

# histogram plot

ax = fig.add_subplot(1, 2, 2)
# compute histogram statistics
h, xedges, yedges = np.histogram2d(phi, psi, bins=[100, 100], range=[[-180, 180],[-180, 180]], density=True)
# get the meshgrid
X, Y = np.meshgrid(xedges, yedges)
# plot the histogram
im = ax.pcolormesh(X, Y, h.T, cmap='coolwarm', shading='auto')

# show colorbar
cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.ax.tick_params(labelsize=15)

ax.set_aspect('equal')
ax.set_xlim([-180, 180])
ax.set_ylim([-180, 180])
ax.set_xlabel(r'$\phi$',fontsize=15)
ax.set_ylabel(r'$\psi$',fontsize=15, rotation=0)
ax.tick_params(axis='both', labelsize=12)
ax.set_xticks([-150, -100, -50, 0, 50, 100, 150])
ax.set_yticks([-150, -100, -50, 0, 50, 100, 150])
ax.set_title('histogram')

plt.show()

### pre-processing 

reshape, shift, and normalize the trajectory data

In [ ]:
traj = traj.reshape(-1, 5 * 3)
traj_mean = traj.mean(axis=0)
traj_std = np.std(traj, axis=0)
traj = (traj - traj_mean) / traj_std

print ('\nShape of trajectory data:', traj.shape)

print ('\nMin and max: ', np.min(traj), np.max(traj))